In [1]:
!pip install --upgrade pip
!pip install inference-gpu supervision
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 15.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting re

In [1]:
from google.colab import userdata
from inference import get_model

/usr/local/lib/python3.13/dist-packages/inference/core/env.py:345: InferenceDeprecationWarning: CORE_MODEL_GAZE_ENABLED is True: POST /gaze/gaze_detection is registered as a deprecation stub returning HTTP 410 Gone. The stub and this flag will be removed end of Q2 2026. Set CORE_MODEL_GAZE_ENABLED=False to disable it now.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[transformers] `DepthProImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `DepthProImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/infer

In [2]:

ROBOFLOW_API_KEY = userdata.get('roboflow_api')
FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/14"
FIELD_DETECTION_MODEL = get_model(model_id=FIELD_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)

Output()

# Installs

In [3]:
!pip install rfdetr pandas opencv-python pillow tqdm numpy torch torchvision trackers -q

# Imports

In [4]:
import cv2
from rfdetr import RFDETRLargeDeprecated
import pandas as pd
import numpy as np
import csv
from dataclasses import dataclass
import cv2
import numpy as np
import supervision as sv
from PIL import Image
import torch
from PIL import Image
from typing import Union, Optional, List, Dict, Tuple
import os
from tqdm import tqdm
import time
import json
from pathlib import Path
import warnings
from huggingface_hub import hf_hub_download
import supervision as sv
import cv2
from google.colab.patches import cv2_imshow
from rfdetr import RFDETRBase


# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Code

In [5]:
model_path = hf_hub_download(repo_id="julianzu9612/RFDETR-Soccernet", filename="weights/checkpoint_best_regular.pth")

weights/checkpoint_best_regular.pth:   0%|          | 0.00/1.57G [00:00<?, ?B/s]

In [6]:
model_path

'/tmp/cache/hf_home/hub/models--julianzu9612--RFDETR-Soccernet/snapshots/1e388b922a64f2be39cbf1925e5fd5fc4f7dd771/weights/checkpoint_best_regular.pth'

In [7]:

device = "cuda" if torch.cuda.is_available() else "cpu"

model = RFDETRLargeDeprecated(
      pretrain_weights=str(model_path),
      num_classes=3,
      device=device,
  )
model.inference(dtype=torch.float16)

# Distance gate

### Ball Tracker

In [8]:
class BallTracker:
    def __init__(
        self,
        base_distance: float,
        max_distance: float,
        velocity_factor: float,
        reacquire_after: int,
        reacquire_confidence: float,
        reacquire_frames: int,
        reacquire_distance: float,
        velocity_smoothing: float,
    ):
        self.base_distance = base_distance
        self.max_distance = max_distance
        self.velocity_factor = velocity_factor

        self.reacquire_after = reacquire_after
        self.reacquire_confidence = reacquire_confidence
        self.reacquire_frames = reacquire_frames
        self.reacquire_distance = reacquire_distance

        self.velocity_smoothing = velocity_smoothing

        self.position = None
        self.velocity = np.zeros(2, dtype=np.float32)

        self.missed_frames = 0

        self.candidate_position = None
        self.candidate_count = 0

    def update(self, detections: sv.Detections):
        if len(detections) == 0:
            self.missed_frames += 1
            return None

        centers = np.column_stack(
            (
                (detections.xyxy[:, 0] + detections.xyxy[:, 2]) / 2,
                (detections.xyxy[:, 1] + detections.xyxy[:, 3]) / 2,
            )
        )

        if self.position is None:
            return self._initialize(
                detections,
                centers,
            )

        frame_gap = self.missed_frames + 1

        predicted_position = (
            self.position
            + self.velocity * frame_gap
        )

        speed = np.linalg.norm(self.velocity)

        allowed_distance = min(
            self.max_distance,
            self.base_distance
            + self.velocity_factor * speed
            + 20 * self.missed_frames,
        )

        distances = np.linalg.norm(
            centers - predicted_position,
            axis=1,
        )

        valid_indices = np.where(
            distances <= allowed_distance
        )[0]

        if len(valid_indices) > 0:
            best_idx = valid_indices[
                np.argmin(distances[valid_indices])
            ]

            if self.missed_frames == 0:
                self._update_position(
                    centers[best_idx]
                )

                self._reset_candidate()

                return int(best_idx)

            return self._confirm_recovery(
                detections,
                centers,
                best_idx,
            )

        self.missed_frames += 1

        if self.missed_frames >= self.reacquire_after:
            return self._global_reacquire(
                detections,
                centers,
            )

        return None

    def _initialize(
        self,
        detections,
        centers,
    ):
        best_idx = np.argmax(
            detections.confidence
        )

        candidate = centers[best_idx]

        if self.candidate_position is None:
            self.candidate_position = candidate
            self.candidate_count = 1

            return None

        distance = np.linalg.norm(
            candidate
            - self.candidate_position
        )

        if distance <= self.reacquire_distance:
            self.candidate_count += 1
            self.candidate_position = candidate
        else:
            self.candidate_position = candidate
            self.candidate_count = 1

        if self.candidate_count < 2:
            return None

        self.position = candidate
        self.velocity[:] = 0
        self.missed_frames = 0

        self._reset_candidate()

        return int(best_idx)

    def _confirm_recovery(
        self,
        detections,
        centers,
        best_idx,
    ):
        candidate = centers[best_idx]

        if self.candidate_position is None:
            self.candidate_position = candidate
            self.candidate_count = 1
            self.missed_frames += 1

            return None

        distance = np.linalg.norm(
            candidate
            - self.candidate_position
        )

        if distance <= self.reacquire_distance:
            self.candidate_position = candidate
            self.candidate_count += 1
        else:
            self.candidate_position = candidate
            self.candidate_count = 1

        if self.candidate_count < self.reacquire_frames:
            self.missed_frames += 1
            return None

        self._update_position(
            candidate
        )

        self._reset_candidate()

        return int(best_idx)

    def _global_reacquire(
        self,
        detections,
        centers,
    ):
        valid_indices = np.where(
            detections.confidence
            >= self.reacquire_confidence
        )[0]

        if len(valid_indices) == 0:
            self._reset_candidate()
            return None

        if self.candidate_position is None:
            best_idx = valid_indices[
                np.argmax(
                    detections.confidence[
                        valid_indices
                    ]
                )
            ]

            self.candidate_position = (
                centers[best_idx]
            )

            self.candidate_count = 1

            return None

        distances = np.linalg.norm(
            centers[valid_indices]
            - self.candidate_position,
            axis=1,
        )

        best_local_idx = np.argmin(
            distances
        )

        best_idx = valid_indices[
            best_local_idx
        ]

        if (
            distances[best_local_idx]
            > self.reacquire_distance
        ):
            best_idx = valid_indices[
                np.argmax(
                    detections.confidence[
                        valid_indices
                    ]
                )
            ]

            self.candidate_position = (
                centers[best_idx]
            )

            self.candidate_count = 1

            return None

        self.candidate_position = (
            centers[best_idx]
        )

        self.candidate_count += 1

        if self.candidate_count < self.reacquire_frames:
            return None

        self.position = (
            self.candidate_position.copy()
        )

        self.velocity[:] = 0
        self.missed_frames = 0

        self._reset_candidate()

        return int(best_idx)

    def _update_position(
        self,
        new_position,
    ):
        frame_gap = self.missed_frames + 1

        measured_velocity = (
            new_position
            - self.position
        ) / frame_gap

        self.velocity = (
            self.velocity_smoothing
            * self.velocity
            + (1 - self.velocity_smoothing)
            * measured_velocity
        )

        self.position = (
            new_position.copy()
        )

        self.missed_frames = 0

    def _reset_candidate(self):
        self.candidate_position = None
        self.candidate_count = 0

### Utils

In [9]:
def get_tile_positions(
    length: int,
    tile_size: int,
    overlap: float,
) -> list[int]:
    if length <= tile_size:
        return [0]

    stride = int(tile_size * (1 - overlap))
    positions = list(range(0, length - tile_size + 1, stride))

    last_position = length - tile_size

    if positions[-1] != last_position:
        positions.append(last_position)

    return positions


def tiled_predict(
    frame: np.ndarray,
    model,
    tile_size: int,
    overlap: float,
    threshold: float,
    nms_threshold: float,
) -> sv.Detections:
    height, width = frame.shape[:2]

    x_positions = get_tile_positions(width, tile_size, overlap)
    y_positions = get_tile_positions(height, tile_size, overlap)

    boxes = []
    confidences = []
    class_ids = []

    for y in y_positions:
        for x in x_positions:
            tile = frame[
                y:min(y + tile_size, height),
                x:min(x + tile_size, width),
            ]

            tile = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)
            tile = Image.fromarray(tile)

            detections = model.predict(
                tile,
                threshold=threshold,
            )

            if len(detections) == 0:
                continue

            tile_boxes = detections.xyxy.copy()
            tile_boxes[:, [0, 2]] += x
            tile_boxes[:, [1, 3]] += y

            boxes.append(tile_boxes)
            confidences.append(detections.confidence)
            class_ids.append(detections.class_id)

    if not boxes:
        return sv.Detections.empty()

    detections = sv.Detections(
        xyxy=np.concatenate(boxes),
        confidence=np.concatenate(confidences),
        class_id=np.concatenate(class_ids),
    )

    return detections.with_nms(
        threshold=nms_threshold,
        class_agnostic=False,
    )


def filter_pitch_detections(
    frame: np.ndarray,
    detections: sv.Detections,
    min_green_ratio: float,
    context_scale: float,
) -> sv.Detections:
    if len(detections) == 0:
        return detections

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_green = np.array([25, 25, 20])
    upper_green = np.array([95, 255, 255])

    green_mask = cv2.inRange(
        hsv,
        lower_green,
        upper_green,
    )

    height, width = frame.shape[:2]
    keep = []

    for x1, y1, x2, y2 in detections.xyxy:
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        box_width = max(x2 - x1, 10) * context_scale
        box_height = max(y2 - y1, 10) * context_scale

        px1 = max(0, int(center_x - box_width / 2))
        py1 = max(0, int(center_y - box_height / 2))
        px2 = min(width, int(center_x + box_width / 2))
        py2 = min(height, int(center_y + box_height / 2))

        patch = green_mask[py1:py2, px1:px2]

        if patch.size == 0:
            keep.append(False)
            continue

        green_ratio = np.mean(patch > 0)

        keep.append(
            green_ratio >= min_green_ratio
        )

    return detections[np.asarray(keep, dtype=bool)]


### Config

In [10]:
@dataclass
class Config:
    input_path: str = "./data/crop1.mp4"
    output_path: str = "output_tiled.mp4"
    csv_path: str = "ball_positions.csv"

    max_frames: int = 9999
    target_class_id: int = 0

    detector_threshold: float = 0.15
    nms_threshold: float = 0.5

    tile_size: int = 640
    tile_overlap: float = 0.25

    use_pitch_filter: bool = True
    pitch_green_min_ratio: float = 0.08
    pitch_context_scale: float = 3.0

    base_distance: float = 80.0
    max_tracking_distance: float = 300.0
    velocity_distance_factor: float = 2.5

    recovery_search_distance: float = 250.0
    missed_distance_factor: float = 30.0

    init_confidence: float = 0.4
    recovery_confidence: float = 0.2
    reacquire_confidence: float = 0.3

    init_frames: int = 3
    recovery_frames: int = 2
    reacquire_frames: int = 2
    reacquire_after: int = 6

    reacquire_distance: float = 120.0

    velocity_smoothing: float = 0.4
    area_smoothing: float = 0.8

    min_area_ratio: float = 0.25
    max_area_ratio: float = 4.0
    max_aspect_ratio: float = 2.0

    distance_weight: float = 0.50
    confidence_weight: float = 0.20
    direction_weight: float = 0.20
    size_weight: float = 0.10

    max_tracking_score: float = 0.75
    direction_min_speed: float = 4.0


### Prediction

In [11]:
import csv
import cv2
import numpy as np
import supervision as sv

from google.colab import userdata
from inference import get_model


ROBOFLOW_API_KEY = userdata.get("roboflow_api")

FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/14"

FIELD_DETECTION_MODEL = get_model(
    model_id=FIELD_DETECTION_MODEL_ID,
    api_key=ROBOFLOW_API_KEY,
)


PITCH_VERTICES = np.array(
    [
        [0.0, 0.0],
        [0.0, 14.5],
        [0.0, 25.84],
        [0.0, 44.16],
        [0.0, 55.5],
        [0.0, 70.0],
        [5.5, 25.84],
        [5.5, 44.16],
        [11.0, 35.0],
        [20.15, 14.5],
        [20.15, 25.84],
        [20.15, 44.16],
        [20.15, 55.5],
        [60.0, 0.0],
        [60.0, 25.85],
        [60.0, 44.15],
        [60.0, 70.0],
        [99.85, 14.5],
        [99.85, 25.84],
        [99.85, 44.16],
        [99.85, 55.5],
        [109.0, 35.0],
        [114.5, 25.84],
        [114.5, 44.16],
        [120.0, 0.0],
        [120.0, 14.5],
        [120.0, 25.84],
        [120.0, 44.16],
        [120.0, 55.5],
        [120.0, 70.0],
        [50.85, 35.0],
        [69.15, 35.0],
    ],
    dtype=np.float32,
)


def estimate_homography(
    frame,
    confidence_threshold=0.5,
    ransac_threshold=0.75,
):
    result = FIELD_DETECTION_MODEL.infer(
        frame,
        confidence=0.3,
    )[0]

    keypoints = sv.KeyPoints.from_inference(result)

    if len(keypoints.xy) == 0:
        return None, 0

    xy = keypoints.xy[0]

    if keypoints.confidence is None:
        return None, 0

    confidence = keypoints.confidence[0]

    if len(xy) != len(PITCH_VERTICES):
        return None, 0

    valid = (
        (confidence >= confidence_threshold)
        & np.isfinite(xy[:, 0])
        & np.isfinite(xy[:, 1])
        & (xy[:, 0] > 0)
        & (xy[:, 1] > 0)
    )

    source_points = xy[valid].astype(np.float32)
    target_points = PITCH_VERTICES[valid].astype(np.float32)

    if len(source_points) < 4:
        return None, 0

    homography, inlier_mask = cv2.findHomography(
        source_points,
        target_points,
        cv2.RANSAC,
        ransac_threshold,
    )

    if homography is None or inlier_mask is None:
        return None, 0

    inlier_count = int(inlier_mask.sum())

    if inlier_count < 4:
        return None, inlier_count

    return homography, inlier_count


def image_to_pitch(
    x,
    y,
    homography,
):
    point = np.array(
        [[[x, y]]],
        dtype=np.float32,
    )

    transformed = cv2.perspectiveTransform(
        point,
        homography,
    )[0, 0]

    return float(transformed[0]), float(transformed[1])


def is_valid_pitch_position(
    x,
    y,
    margin=5.0,
):
    return (
        -margin <= x <= 120.0 + margin
        and -margin <= y <= 70.0 + margin
    )


cfg = Config()

cap = cv2.VideoCapture(cfg.input_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    cfg.output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

ball_tracker = BallTracker(
    base_distance=80,
    max_distance=300,
    velocity_factor=2.5,
    reacquire_after=6,
    reacquire_confidence=0.30,
    reacquire_frames=2,
    reacquire_distance=120,
    velocity_smoothing=0.4,
)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

frame_num = 0

current_homography = None
homography_missing_frames = 0
max_homography_missing_frames = 10

with open(cfg.csv_path, "w", newline="") as csv_file:
    csv_writer = csv.writer(csv_file)

    csv_writer.writerow(
        [
            "frame",
            "timestamp",
            "image_x",
            "image_y",
            "pitch_x",
            "pitch_y",
            "confidence",
            "homography_inliers",
        ]
    )

    while cap.isOpened() and frame_num < cfg.max_frames:
        ret, frame = cap.read()

        if not ret:
            break

        new_homography, homography_inliers = estimate_homography(
            frame,
        )

        if new_homography is not None:
            current_homography = new_homography
            homography_missing_frames = 0
        else:
            homography_missing_frames += 1

            if homography_missing_frames > max_homography_missing_frames:
                current_homography = None

        detections = tiled_predict(
            frame=frame,
            model=model,
            tile_size=cfg.tile_size,
            overlap=cfg.tile_overlap,
            threshold=cfg.detector_threshold,
            nms_threshold=cfg.nms_threshold,
        )

        detections = detections[
            detections.class_id == cfg.target_class_id
        ]

        ball_idx = ball_tracker.update(
            detections
        )

        timestamp = frame_num / fps
        annotated_frame = frame.copy()

        if ball_idx is None:
            csv_writer.writerow(
                [
                    frame_num,
                    timestamp,
                    "",
                    "",
                    "",
                    "",
                    "",
                    homography_inliers,
                ]
            )

        else:
            ball = detections[[ball_idx]]

            x1, y1, x2, y2 = ball.xyxy[0]

            ball_x = float((x1 + x2) / 2)
            ball_y = float((y1 + y2) / 2)

            confidence = float(
                ball.confidence[0]
            )

            pitch_x = None
            pitch_y = None

            if current_homography is not None:
                transformed_x, transformed_y = image_to_pitch(
                    ball_x,
                    ball_y,
                    current_homography,
                )

                if is_valid_pitch_position(
                    transformed_x,
                    transformed_y,
                ):
                    pitch_x = transformed_x
                    pitch_y = transformed_y

            annotated_frame = box_annotator.annotate(
                annotated_frame,
                ball,
            )

            label = f"Ball {confidence:.2f}"

            if pitch_x is not None:
                label += (
                    f" | pitch "
                    f"({pitch_x:.1f}, {pitch_y:.1f}) m"
                )

            annotated_frame = label_annotator.annotate(
                annotated_frame,
                ball,
                labels=[label],
            )

            cv2.circle(
                annotated_frame,
                (
                    int(ball_x),
                    int(ball_y),
                ),
                6,
                (0, 0, 255),
                -1,
            )

            csv_writer.writerow(
                [
                    frame_num,
                    timestamp,
                    ball_x,
                    ball_y,
                    "" if pitch_x is None else pitch_x,
                    "" if pitch_y is None else pitch_y,
                    confidence,
                    homography_inliers,
                ]
            )

        writer.write(
            annotated_frame
        )

        frame_num += 1

cap.release()
writer.release()

In [14]:
from google.colab import files
files.download('./output_tiled.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import csv

import cv2
import numpy as np
import supervision as sv

from google.colab import userdata
from inference import get_model


ROBOFLOW_API_KEY = userdata.get("roboflow_api")

FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/14"

FIELD_DETECTION_MODEL = get_model(
    model_id=FIELD_DETECTION_MODEL_ID,
    api_key=ROBOFLOW_API_KEY,
)

PITCH_WIDTH = 120.0
PITCH_HEIGHT = 70.0

PITCH_VERTICES = np.array(
    [
        [0.0, 0.0],
        [0.0, 14.5],
        [0.0, 25.84],
        [0.0, 44.16],
        [0.0, 55.5],
        [0.0, 70.0],
        [5.5, 25.84],
        [5.5, 44.16],
        [11.0, 35.0],
        [20.15, 14.5],
        [20.15, 25.84],
        [20.15, 44.16],
        [20.15, 55.5],
        [60.0, 0.0],
        [60.0, 25.85],
        [60.0, 44.15],
        [60.0, 70.0],
        [99.85, 14.5],
        [99.85, 25.84],
        [99.85, 44.16],
        [99.85, 55.5],
        [109.0, 35.0],
        [114.5, 25.84],
        [114.5, 44.16],
        [120.0, 0.0],
        [120.0, 14.5],
        [120.0, 25.84],
        [120.0, 44.16],
        [120.0, 55.5],
        [120.0, 70.0],
        [50.85, 35.0],
        [69.15, 35.0],
    ],
    dtype=np.float32,
)


def estimate_homography(
    frame,
    confidence_threshold=0.5,
    ransac_threshold=0.75,
):
    result = FIELD_DETECTION_MODEL.infer(
        frame,
        confidence=0.3,
    )[0]

    keypoints = sv.KeyPoints.from_inference(result)

    if len(keypoints.xy) == 0:
        return None, 0

    if keypoints.confidence is None:
        return None, 0

    xy = keypoints.xy[0]
    confidence = keypoints.confidence[0]

    if len(xy) != len(PITCH_VERTICES):
        return None, 0

    valid = (
        (confidence >= confidence_threshold)
        & np.isfinite(xy[:, 0])
        & np.isfinite(xy[:, 1])
        & (xy[:, 0] > 0)
        & (xy[:, 1] > 0)
    )

    source_points = xy[valid].astype(np.float32)
    target_points = PITCH_VERTICES[valid].astype(np.float32)

    if len(source_points) < 4:
        return None, 0

    homography, inlier_mask = cv2.findHomography(
        source_points,
        target_points,
        cv2.RANSAC,
        ransac_threshold,
    )

    if homography is None or inlier_mask is None:
        return None, 0

    inlier_count = int(inlier_mask.sum())

    if inlier_count < 4:
        return None, inlier_count

    return homography, inlier_count


def image_to_pitch(
    x,
    y,
    homography,
):
    point = np.array(
        [[[x, y]]],
        dtype=np.float32,
    )

    transformed = cv2.perspectiveTransform(
        point,
        homography,
    )[0, 0]

    return float(transformed[0]), float(transformed[1])


def is_valid_pitch_position(
    x,
    y,
    margin=5.0,
):
    return (
        -margin <= x <= PITCH_WIDTH + margin
        and -margin <= y <= PITCH_HEIGHT + margin
    )


def create_pitch_map(
    width=360,
    height=210,
):
    pitch = np.zeros(
        (height, width, 3),
        dtype=np.uint8,
    )

    pitch[:] = (50, 120, 50)

    margin = 8
    line_color = (255, 255, 255)
    thickness = 2

    left = margin
    right = width - margin
    top = margin
    bottom = height - margin

    usable_width = right - left
    usable_height = bottom - top

    cv2.rectangle(
        pitch,
        (left, top),
        (right, bottom),
        line_color,
        thickness,
    )

    center_x = left + usable_width // 2
    center_y = top + usable_height // 2

    cv2.line(
        pitch,
        (center_x, top),
        (center_x, bottom),
        line_color,
        thickness,
    )

    center_circle_radius = int(
        usable_height * 9.15 / PITCH_HEIGHT
    )

    cv2.circle(
        pitch,
        (center_x, center_y),
        center_circle_radius,
        line_color,
        thickness,
    )

    cv2.circle(
        pitch,
        (center_x, center_y),
        3,
        line_color,
        -1,
    )

    penalty_depth = int(
        usable_width * 16.5 / PITCH_WIDTH
    )

    penalty_height = int(
        usable_height * 40.32 / PITCH_HEIGHT
    )

    penalty_top = center_y - penalty_height // 2
    penalty_bottom = center_y + penalty_height // 2

    cv2.rectangle(
        pitch,
        (left, penalty_top),
        (left + penalty_depth, penalty_bottom),
        line_color,
        thickness,
    )

    cv2.rectangle(
        pitch,
        (right - penalty_depth, penalty_top),
        (right, penalty_bottom),
        line_color,
        thickness,
    )

    goal_area_depth = int(
        usable_width * 5.5 / PITCH_WIDTH
    )

    goal_area_height = int(
        usable_height * 18.32 / PITCH_HEIGHT
    )

    goal_area_top = center_y - goal_area_height // 2
    goal_area_bottom = center_y + goal_area_height // 2

    cv2.rectangle(
        pitch,
        (left, goal_area_top),
        (left + goal_area_depth, goal_area_bottom),
        line_color,
        thickness,
    )

    cv2.rectangle(
        pitch,
        (right - goal_area_depth, goal_area_top),
        (right, goal_area_bottom),
        line_color,
        thickness,
    )

    left_penalty_x = left + int(
        usable_width * 11.0 / PITCH_WIDTH
    )

    right_penalty_x = right - int(
        usable_width * 11.0 / PITCH_WIDTH
    )

    cv2.circle(
        pitch,
        (left_penalty_x, center_y),
        3,
        line_color,
        -1,
    )

    cv2.circle(
        pitch,
        (right_penalty_x, center_y),
        3,
        line_color,
        -1,
    )

    return pitch


def pitch_position_to_map(
    pitch_x,
    pitch_y,
    map_width,
    map_height,
):
    margin = 8

    usable_width = map_width - 2 * margin
    usable_height = map_height - 2 * margin

    normalized_x = np.clip(
        pitch_x / PITCH_WIDTH,
        0.0,
        1.0,
    )

    normalized_y = np.clip(
        pitch_y / PITCH_HEIGHT,
        0.0,
        1.0,
    )

    map_x = int(
        margin + normalized_x * usable_width
    )

    map_y = int(
        margin + normalized_y * usable_height
    )

    return map_x, map_y


def add_pitch_overlay(
    frame,
    pitch_x=None,
    pitch_y=None,
    map_width=360,
    map_height=210,
    margin=20,
    opacity=0.85,
):
    frame_height, frame_width = frame.shape[:2]

    max_width = frame_width - 2 * margin
    max_height = frame_height - 2 * margin

    if map_width > max_width:
        scale = max_width / map_width
        map_width = int(map_width * scale)
        map_height = int(map_height * scale)

    if map_height > max_height:
        scale = max_height / map_height
        map_width = int(map_width * scale)
        map_height = int(map_height * scale)

    pitch_map = create_pitch_map(
        width=map_width,
        height=map_height,
    )

    if pitch_x is not None and pitch_y is not None:
        ball_x, ball_y = pitch_position_to_map(
            pitch_x,
            pitch_y,
            map_width,
            map_height,
        )

        cv2.circle(
            pitch_map,
            (ball_x, ball_y),
            8,
            (255, 255, 255),
            -1,
        )

        cv2.circle(
            pitch_map,
            (ball_x, ball_y),
            6,
            (0, 0, 255),
            -1,
        )

    x1 = margin
    y1 = frame_height - map_height - margin

    x2 = x1 + map_width
    y2 = y1 + map_height

    roi = frame[y1:y2, x1:x2]

    blended = cv2.addWeighted(
        roi,
        1.0 - opacity,
        pitch_map,
        opacity,
        0,
    )

    frame[y1:y2, x1:x2] = blended

    cv2.rectangle(
        frame,
        (x1 - 2, y1 - 2),
        (x2 + 2, y2 + 2),
        (255, 255, 255),
        2,
    )

    return frame


cfg = Config()

cap = cv2.VideoCapture(cfg.input_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    cfg.output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

ball_tracker = BallTracker(
    base_distance=80,
    max_distance=300,
    velocity_factor=2.5,
    reacquire_after=6,
    reacquire_confidence=0.30,
    reacquire_frames=2,
    reacquire_distance=120,
    velocity_smoothing=0.4,
)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

frame_num = 0

current_homography = None
homography_missing_frames = 0
max_homography_missing_frames = 10

with open(cfg.csv_path, "w", newline="") as csv_file:
    csv_writer = csv.writer(csv_file)

    csv_writer.writerow(
        [
            "frame",
            "timestamp",
            "image_x",
            "image_y",
            "pitch_x",
            "pitch_y",
            "confidence",
            "homography_inliers",
        ]
    )

    while cap.isOpened() and frame_num < cfg.max_frames:
        ret, frame = cap.read()

        if not ret:
            break

        new_homography, homography_inliers = estimate_homography(
            frame,
        )

        if new_homography is not None:
            current_homography = new_homography
            homography_missing_frames = 0
        else:
            homography_missing_frames += 1

            if homography_missing_frames > max_homography_missing_frames:
                current_homography = None

        detections = tiled_predict(
            frame=frame,
            model=model,
            tile_size=cfg.tile_size,
            overlap=cfg.tile_overlap,
            threshold=cfg.detector_threshold,
            nms_threshold=cfg.nms_threshold,
        )

        detections = detections[
            detections.class_id == cfg.target_class_id
        ]

        ball_idx = ball_tracker.update(
            detections
        )

        timestamp = frame_num / fps
        annotated_frame = frame.copy()

        pitch_x = None
        pitch_y = None

        if ball_idx is None:
            csv_writer.writerow(
                [
                    frame_num,
                    timestamp,
                    "",
                    "",
                    "",
                    "",
                    "",
                    homography_inliers,
                ]
            )

        else:
            ball = detections[[ball_idx]]

            x1, y1, x2, y2 = ball.xyxy[0]

            ball_x = float((x1 + x2) / 2)
            ball_y = float((y1 + y2) / 2)

            confidence = float(
                ball.confidence[0]
            )

            if current_homography is not None:
                transformed_x, transformed_y = image_to_pitch(
                    ball_x,
                    ball_y,
                    current_homography,
                )

                if is_valid_pitch_position(
                    transformed_x,
                    transformed_y,
                ):
                    pitch_x = transformed_x
                    pitch_y = transformed_y

            annotated_frame = box_annotator.annotate(
                annotated_frame,
                ball,
            )

            label = f"Ball {confidence:.2f}"

            if pitch_x is not None:
                label += (
                    f" | ({pitch_x:.1f}, "
                    f"{pitch_y:.1f}) m"
                )

            annotated_frame = label_annotator.annotate(
                annotated_frame,
                ball,
                labels=[label],
            )

            cv2.circle(
                annotated_frame,
                (
                    int(ball_x),
                    int(ball_y),
                ),
                6,
                (0, 0, 255),
                -1,
            )

            csv_writer.writerow(
                [
                    frame_num,
                    timestamp,
                    ball_x,
                    ball_y,
                    "" if pitch_x is None else pitch_x,
                    "" if pitch_y is None else pitch_y,
                    confidence,
                    homography_inliers,
                ]
            )

        annotated_frame = add_pitch_overlay(
            annotated_frame,
            pitch_x=pitch_x,
            pitch_y=pitch_y,
            map_width=360,
            map_height=210,
            margin=20,
            opacity=0.85,
        )

        writer.write(
            annotated_frame
        )

        frame_num += 1

cap.release()
writer.release()